In [35]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from umap import UMAP
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import skew, kurtosis
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Define output directory
OUTPUT_DIR = Path.cwd() / 'users_final_demo99999'
OUTPUT_DIR.mkdir(exist_ok=True)

### Helper Functions

def save_df(df, name):
    """Save a DataFrame to the output directory."""
    path = OUTPUT_DIR / name
    df.to_csv(path, index=False)
    logging.info(f"Saved DataFrame to {path}")
    return path

def preprocess_cinemate(cinemate_path):
    """Preprocess CinemateMovieDataset for TMDB compatibility."""
    logging.info(f"Preprocessing {cinemate_path}")
    movies = pd.read_csv(cinemate_path, dtype={'TMDBId': 'Int64', 'GenreId': str})
    logging.info(f"Loaded {len(movies)} rows from Cinemate dataset")
    movies['genres'] = movies.groupby('TMDBId')['GenreId'].transform(lambda x: '|'.join(x))
    movies = movies.drop_duplicates(subset='TMDBId')
    movies['Year'] = pd.to_datetime(movies['ReleaseDate'], errors='coerce').dt.year
    movies = movies[movies['genres'].str.strip() != '']
    logging.info(f"After preprocessing, {len(movies)} movies remain")
    return movies[['TMDBId', 'Title', 'Year', 'genres']]

def create_mapping(links_path, movies_ml_path):
    """Create mapping from MovieLens MovieID to TMDBId using ml-25m links.csv."""
    logging.info(f"Creating mapping from {links_path} and {movies_ml_path}")
    links = pd.read_csv(links_path, dtype={'movieId': str, 'tmdbId': str})
    links['tmdbId'] = pd.to_numeric(links['tmdbId'], errors='coerce').astype('Int64')
    links = links.dropna(subset=['tmdbId']).rename(columns={'movieId': 'MovieID', 'tmdbId': 'TMDBId'})
    links['MovieID'] = pd.to_numeric(links['MovieID'], errors='coerce').astype('Int64')
    logging.info(f"Loaded {len(links)} links after cleaning")

    movies_ml = pd.read_csv(movies_ml_path, sep='::', engine='python', 
                            names=['MovieID', 'Title', 'Genres'], encoding='latin-1',
                            dtype={'MovieID': str, 'Title': str, 'Genres': str})
    movies_ml['MovieID'] = pd.to_numeric(movies_ml['MovieID'], errors='coerce').astype('Int64')
    logging.info(f"Loaded {len(movies_ml)} movies from MovieLens")

    mapping = movies_ml[['MovieID']].merge(links[['MovieID', 'TMDBId']], on='MovieID', how='inner')
    stats = {
        'total_movies': len(movies_ml),
        'mapped_movies': len(mapping),
        'dropped_movies': len(movies_ml) - len(mapping)
    }
    logging.info(f"Mapping stats: {stats}")
    return mapping, stats

def load_data(ratings_path, users_path, movies_cinemate_path, mapping_path):
    """Load and preprocess ratings, users, and movies with TMDBId mapping."""
    logging.info(f"Loading ratings from {ratings_path}")
    ratings = pd.read_csv(ratings_path, sep='::', engine='python', 
                          names=['userId', 'movieId', 'rating'], 
                          usecols=['userId', 'movieId', 'rating'], 
                          dtype={'userId': str, 'movieId': str, 'rating': float})
    ratings['userId'] = pd.to_numeric(ratings['userId'], errors='coerce').astype('Int64')
    ratings['movieId'] = pd.to_numeric(ratings['movieId'], errors='coerce').astype('Int64')
    ratings = ratings.dropna(subset=['userId', 'movieId'])
    logging.info(f"Loaded {len(ratings)} ratings for {ratings['userId'].nunique()} users")
    logging.info(f"Ratings movieId range: {ratings['movieId'].min()} to {ratings['movieId'].max()}")
    logging.info(f"Ratings rating range: {ratings['rating'].min()} to {ratings['rating'].max()}")
    logging.info(f"Sample ratings:\n{ratings.head(5).to_string()}")

    age_map = {1: 16, 18: 21, 25: 29.5, 35: 39.5, 45: 47, 50: 52.5, 56: 60}
    logging.info(f"Loading users from {users_path}")
    users = pd.read_csv(users_path, sep='::', engine='python', 
                        names=['userId', 'gender', 'age', 'occupation'], 
                        usecols=['userId', 'gender', 'age', 'occupation'], 
                        dtype={'userId': str, 'gender': str, 'age': str, 'occupation': str})
    users['userId'] = pd.to_numeric(users['userId'], errors='coerce').astype('Int64')
    users['age'] = pd.to_numeric(users['age'], errors='coerce')
    logging.info(f"Unique age values before mapping: {users['age'].unique()}")
    users['age'] = users['age'].map(age_map).fillna(29.5).astype('float32')
    logging.info(f"Unique age values after mapping: {users['age'].unique()}")
    users = users.dropna(subset=['userId'])
    logging.info(f"Loaded {len(users)} users")

    logging.info(f"Loading movies from {movies_cinemate_path}")
    movies = pd.read_csv(movies_cinemate_path, 
                         usecols=['TMDBId', 'Title', 'genres'], 
                         dtype={'TMDBId': 'Int64', 'Title': str, 'genres': str})
    movies = movies.rename(columns={'TMDBId': 'movieId'})
    logging.info(f"Loaded {len(movies)} movies")
    logging.info(f"Movies movieId range: {movies['movieId'].min()} to {movies['movieId'].max()}")

    logging.info(f"Loading mapping from {mapping_path}")
    mapping = pd.read_csv(mapping_path, dtype={'MovieID': 'Int64', 'TMDBId': 'Int64'})
    logging.info(f"Loaded mapping with {len(mapping)} entries")

    original_ratings_count = len(ratings)
    original_users = ratings['userId'].nunique()
    ratings = ratings.merge(mapping, left_on='movieId', right_on='MovieID', how='inner')
    ratings = ratings.drop(columns=['movieId', 'MovieID']).rename(columns={'TMDBId': 'movieId'})
    logging.info(f"After merging with mapping, {len(ratings)} ratings remain for {ratings['userId'].nunique()} users")
    logging.info(f"Merged ratings movieId range: {ratings['movieId'].min()} to {ratings['movieId'].max()}")

    stats = {
        'total_ratings': original_ratings_count,
        'mapped_ratings': len(ratings),
        'dropped_ratings': original_ratings_count - len(ratings),
        'dropped_ratings_pct': (original_ratings_count - len(ratings)) / original_ratings_count,
        'total_users': original_users,
        'mapped_users': ratings['userId'].nunique(),
        'dropped_users': original_users - ratings['userId'].nunique()
    }
    return ratings, movies, users, stats

def normalize_ratings(ratings):
    """Normalize ratings per user using z-score."""
    logging.info("Normalizing ratings")
    stats = ratings.groupby('userId')['rating'].agg(['mean', 'std']).fillna({'std': 1})
    stats['std'] = stats['std'].where(stats['std'] != 0, 1)
    merged = ratings.merge(stats, on='userId')
    merged['norm_rating'] = (merged['rating'] - merged['mean']) / merged['std']
    normalized = merged[['userId', 'movieId', 'norm_rating', 'rating']]
    logging.info(f"Normalized {len(normalized)} ratings")
    logging.info(f"Normalized rating range: {normalized['rating'].min()} to {normalized['rating'].max()}")
    return normalized

def build_features(ratings_norm, movies, users, min_ratings=50, genre_weight=0.4, demo_weight=0.3, rating_weight=0.3):
    """Build user features: genre TF-IDF, demographics, and enhanced rating statistics."""
    logging.info("Building features")
    df = ratings_norm.merge(movies, on='movieId', how='left')
    df['genres'] = df['genres'].fillna('')
    df = df[df['genres'].str.strip() != '']
    logging.info(f"After merging ratings with movies, {len(df)} ratings remain")
    exploded = df.assign(genres=df['genres'].str.split('|')).explode('genres')
    genre_mean = exploded.groupby(['userId', 'genres'])['norm_rating'].mean().unstack(fill_value=0)
    counts = ratings_norm['userId'].value_counts()
    valid = counts[counts >= min_ratings].index
    genre_mean = genre_mean.loc[valid]
    logging.info(f"{len(valid)} users have at least {min_ratings} ratings")
    N = len(genre_mean)
    idf = np.log((N + 3) / (3 + (genre_mean != 0).sum())) + 1
    genre_tfidf = genre_mean * idf
    scaler = StandardScaler()
    genre_scaled = pd.DataFrame(scaler.fit_transform(genre_tfidf), 
                                index=genre_tfidf.index, columns=genre_tfidf.columns)
    genre_scaled *= genre_weight
    logging.info(f"Generated genre TF-IDF features for {len(genre_scaled)} users")

    demo = users[users['userId'].isin(valid)].copy()
    demo = pd.get_dummies(demo, columns=['gender', 'occupation'], dtype=float)
    demo['age'] = demo['age'].astype(float)
    demo_index = demo['userId']
    demo_feat = demo.drop(columns=['userId'])
    demo_scaled = pd.DataFrame(scaler.fit_transform(demo_feat), 
                               index=demo_index, columns=demo_feat.columns)
    demo_scaled *= demo_weight
    logging.info(f"Generated demographic features for {len(demo_scaled)} users")

    rating_stats = ratings_norm.groupby('userId').agg({
        'rating': ['mean', 'std', lambda x: skew(x, nan_policy='omit'), 
                   lambda x: kurtosis(x, nan_policy='omit')],
        'movieId': 'count'
    }).fillna(0)
    rating_stats.columns = ['rating_mean', 'rating_std', 'rating_skew', 'rating_kurtosis', 'rating_count']
    rating_stats = pd.DataFrame(scaler.fit_transform(rating_stats), 
                                index=rating_stats.index, 
                                columns=rating_stats.columns)
    rating_stats *= rating_weight
    logging.info(f"Generated rating statistics for {len(rating_stats)} users")

    features = genre_scaled.join([demo_scaled, rating_stats], how='left').fillna(0)
    logging.info(f"Combined features for {len(features)} users")
    return features, valid

def reduce_dim(features, n_components=3):
    """Reduce dimensionality using UMAP."""
    logging.info("Reducing dimensionality with UMAP")
    reducer = UMAP(n_components=n_components, random_state=42, n_neighbors=15, n_jobs=1)
    reduced = reducer.fit_transform(features)
    logging.info(f"Reduced to {reduced.shape} with UMAP")
    return reduced

def cluster_users(reduced, n_clusters=5):
    """Cluster users using KMeans."""
    logging.info(f"Clustering with {n_clusters} clusters")
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
    return kmeans.fit_predict(reduced)

def evaluate_and_plot(reduced, labels, valid_users, n_clusters):
    """Evaluate clustering, save scatter plot, and compute balance penalty."""
    logging.info(f"Evaluating {n_clusters} clusters")
    silhouette = silhouette_score(reduced, labels)
    cluster_sizes = pd.Series(labels).value_counts().sort_index()
    balance_penalty = cluster_sizes.std() / cluster_sizes.mean() if len(cluster_sizes) > 1 else 0
    adjusted_score = silhouette - 0.1 * balance_penalty
    
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x=reduced[:, 0], y=reduced[:, 1], hue=labels, palette='viridis', s=50)
    plt.xlabel('UMAP Component 1')
    plt.ylabel('UMAP Component 2')
    plt.title(f'Clusters (n={n_clusters}, Silhouette: {silhouette:.3f}, Balance Penalty: {balance_penalty:.3f})')
    plot_path = OUTPUT_DIR / f'cluster_plot_n{n_clusters}.png'
    plt.savefig(plot_path, bbox_inches='tight')
    plt.close()
    logging.info(f"Saved cluster plot to {plot_path}")
    
    return silhouette, adjusted_score, cluster_sizes, plot_path

def tune_clusters(features, valid_users, min_clusters=5, max_clusters=15):
    """Tune number of clusters to maximize adjusted silhouette score."""
    logging.info(f"Tuning clusters from {min_clusters} to {max_clusters}")
    best_score = -1
    best_n = min_clusters
    best_labels = None
    best_stats = None
    reduced = reduce_dim(features)

    for n in range(min_clusters, max_clusters + 1):
        labels = cluster_users(reduced, n_clusters=n)
        silhouette, adjusted_score, cluster_sizes, plot_path = evaluate_and_plot(
            reduced, labels, valid_users, n)
        print(f"Clusters: {n}, Silhouette: {silhouette:.3f}, Adjusted Score: {adjusted_score:.3f}")
        print(f"Cluster sizes: {cluster_sizes.to_dict()}")
        if adjusted_score > best_score:
            best_score = adjusted_score
            best_n = n
            best_labels = labels
            best_stats = {'silhouette': silhouette, 'adjusted_score': adjusted_score, 
                          'cluster_sizes': cluster_sizes, 'plot_path': plot_path}

    return best_n, best_labels, best_stats

def save_cluster_profiles(cluster_df, ratings_norm, movies, users):
    """Save cluster profiles with average rating stats and demographics."""
    logging.info("Saving cluster profiles")
    cluster_ratings = cluster_df.merge(ratings_norm, on='userId').groupby('cluster').agg({
        'rating': ['mean', 'std', 'count']
    }).fillna(0)
    cluster_ratings.columns = ['avg_rating', 'rating_std', 'rating_count']
    
    cluster_demo = cluster_df.merge(users, on='userId').groupby('cluster').agg({
        'age': 'mean',
        'gender': lambda x: x.mode().iloc[0] if not x.empty else 'Unknown',
        'occupation': lambda x: x.mode().iloc[0] if not x.empty else 'Unknown'
    })
    
    profiles = cluster_ratings.join(cluster_demo, how='left')
    save_df(profiles, 'cluster_profiles.csv')
    return profiles

### Main Execution

if __name__ == '__main__':
    # File paths
    CINEMATE_PATH = r'C:\Users\PC\Downloads\CinemateMovieDataset (1).csv'
    RATINGS_PATH = r'C:\Users\PC\Downloads\ml-1m\ml-1m\ratings.dat'
    USERS_PATH = r'C:\Users\PC\Downloads\ml-1m\ml-1m\users.dat'
    MOVIES_PATH = r'C:\Users\PC\Downloads\ml-1m\ml-1m\movies.dat'
    LINKS_PATH = r'C:\Users\PC\Downloads\ml-25m\ml-25m\links.csv'

    # Verify file existence
    for path in [CINEMATE_PATH, RATINGS_PATH, USERS_PATH, MOVIES_PATH, LINKS_PATH]:
        if not Path(path).exists():
            logging.error(f"File not found: {path}")
            raise FileNotFoundError(f"File not found: {path}")

    # Preprocess Cinemate dataset
    movies_cinemate = preprocess_cinemate(CINEMATE_PATH)
    cinemate_path = save_df(movies_cinemate, 'movies_cinemate_preprocessed.csv')

    # Create MovieID to TMDBId mapping
    mapping, map_stats = create_mapping(LINKS_PATH, MOVIES_PATH)
    mapping_path = save_df(mapping, 'mapping.csv')

    # Load data
    ratings, movies, users, load_stats = load_data(RATINGS_PATH, USERS_PATH, cinemate_path, mapping_path)

    # Print statistics
    print(f"Mapping: {map_stats['mapped_movies']}/{map_stats['total_movies']} movies mapped "
          f"({map_stats['dropped_movies']} dropped, {map_stats['dropped_movies']/map_stats['total_movies']:.2%})")
    print(f"Ratings: {load_stats['mapped_ratings']}/{load_stats['total_ratings']} ratings mapped "
          f"({load_stats['dropped_ratings']} dropped, {load_stats['dropped_ratings_pct']:.2%})")
    print(f"Users: {load_stats['mapped_users']}/{load_stats['total_users']} users mapped "
          f"({load_stats['dropped_users']} dropped, {load_stats['dropped_users'] / load_stats['total_users']:.2%})")

    # Normalize ratings
    ratings_norm = normalize_ratings(ratings)

    # Build features with enhanced rating statistics
    features, valid_users = build_features(ratings_norm, movies, users, 
                                          min_ratings=50, genre_weight=0.4, 
                                          demo_weight=0.3, rating_weight=0.3)

    # Tune clusters
    best_n, best_labels, best_stats = tune_clusters(features, valid_users)

    # Print final results
    print(f"\nBest number of clusters: {best_n}")
    print(f"Silhouette Score: {best_stats['silhouette']:.3f}")
    print(f"Adjusted Score: {best_stats['adjusted_score']:.3f}")
    print(f"Cluster sizes: {best_stats['cluster_sizes'].to_dict()}")
    print(f"Cluster plot saved to: {best_stats['plot_path']}")

    # Save cluster assignments
    cluster_df = pd.DataFrame({'userId': valid_users, 'cluster': best_labels})
    save_df(cluster_df, 'user_clusters.csv')

    # Save cluster profiles
    profiles = save_cluster_profiles(cluster_df, ratings_norm, movies, users)
    print(f"Cluster profiles saved to: {OUTPUT_DIR}/cluster_profiles.csv")

2025-06-08 18:39:27,676 - INFO - Preprocessing C:\Users\PC\Downloads\CinemateMovieDataset (1).csv
2025-06-08 18:39:28,442 - INFO - Loaded 173984 rows from Cinemate dataset
2025-06-08 18:39:34,108 - INFO - After preprocessing, 83860 movies remain
2025-06-08 18:39:34,371 - INFO - Saved DataFrame to C:\Users\PC\users_final_demo99999\movies_cinemate_preprocessed.csv
2025-06-08 18:39:34,386 - INFO - Creating mapping from C:\Users\PC\Downloads\ml-25m\ml-25m\links.csv and C:\Users\PC\Downloads\ml-1m\ml-1m\movies.dat
2025-06-08 18:39:34,516 - INFO - Loaded 62316 links after cleaning
2025-06-08 18:39:34,531 - INFO - Loaded 3883 movies from MovieLens
2025-06-08 18:39:34,531 - INFO - Mapping stats: {'total_movies': 3883, 'mapped_movies': 3831, 'dropped_movies': 52}
2025-06-08 18:39:34,539 - INFO - Saved DataFrame to C:\Users\PC\users_final_demo99999\mapping.csv
2025-06-08 18:39:34,539 - INFO - Loading ratings from C:\Users\PC\Downloads\ml-1m\ml-1m\ratings.dat
2025-06-08 18:39:39,077 - INFO - Load

Mapping: 3831/3883 movies mapped (52 dropped, 1.34%)
Ratings: 997136/1000209 ratings mapped (3073 dropped, 0.31%)
Users: 6040/6040 users mapped (0 dropped, 0.00%)


2025-06-08 18:39:39,860 - INFO - After merging ratings with movies, 989769 ratings remain
2025-06-08 18:39:42,791 - INFO - 4289 users have at least 50 ratings
2025-06-08 18:39:42,809 - INFO - Generated genre TF-IDF features for 4289 users
2025-06-08 18:39:42,809 - INFO - Generated demographic features for 4289 users
2025-06-08 18:39:47,050 - INFO - Generated rating statistics for 6040 users
2025-06-08 18:39:47,055 - INFO - Combined features for 4289 users
2025-06-08 18:39:47,138 - INFO - Tuning clusters from 5 to 15
2025-06-08 18:39:47,138 - INFO - Reducing dimensionality with UMAP
2025-06-08 18:39:53,923 - INFO - Reduced to (4289, 3) with UMAP
2025-06-08 18:39:53,926 - INFO - Clustering with 5 clusters
2025-06-08 18:39:53,980 - INFO - Evaluating 5 clusters
2025-06-08 18:39:54,665 - INFO - Saved cluster plot to C:\Users\PC\users_final_demo99999\cluster_plot_n5.png
2025-06-08 18:39:54,665 - INFO - Clustering with 6 clusters
2025-06-08 18:39:54,730 - INFO - Evaluating 6 clusters


Clusters: 5, Silhouette: 0.514, Adjusted Score: 0.417
Cluster sizes: {0: 510, 1: 2335, 2: 631, 3: 459, 4: 354}


2025-06-08 18:39:55,399 - INFO - Saved cluster plot to C:\Users\PC\users_final_demo99999\cluster_plot_n6.png
2025-06-08 18:39:55,400 - INFO - Clustering with 7 clusters
2025-06-08 18:39:55,473 - INFO - Evaluating 7 clusters


...

Clusters: 13, Silhouette: 0.700, Adjusted Score: 0.629
Cluster sizes: {0: 340, 1: 220, 2: 299, 3: 364, 4: 290, 5: 342, 6: 332, 7: 1001, 8: 218, 9: 84, 10: 136, 11: 117, 12: 546}


2025-06-08 18:40:00,945 - INFO - Saved cluster plot to C:\Users\PC\users_final_demo99999\cluster_plot_n14.png
2025-06-08 18:40:00,945 - INFO - Clustering with 15 clusters
2025-06-08 18:40:01,079 - INFO - Evaluating 15 clusters


Clusters: 14, Silhouette: 0.713, Adjusted Score: 0.636
Cluster sizes: {0: 186, 1: 1000, 2: 342, 3: 290, 4: 299, 5: 364, 6: 136, 7: 117, 8: 84, 9: 340, 10: 547, 11: 220, 12: 218, 13: 146}


2025-06-08 18:40:01,613 - INFO - Saved cluster plot to C:\Users\PC\users_final_demo99999\cluster_plot_n15.png
2025-06-08 18:40:01,629 - INFO - Saved DataFrame to C:\Users\PC\users_final_demo99999\user_clusters.csv
2025-06-08 18:40:01,629 - INFO - Saving cluster profiles
2025-06-08 18:40:01,763 - INFO - Saved DataFrame to C:\Users\PC\users_final_demo99999\cluster_profiles.csv


Clusters: 15, Silhouette: 0.730, Adjusted Score: 0.647
Cluster sizes: {0: 294, 1: 1000, 2: 547, 3: 299, 4: 84, 5: 101, 6: 364, 7: 290, 8: 136, 9: 220, 10: 117, 11: 218, 12: 340, 13: 231, 14: 48}

Best number of clusters: 15
Silhouette Score: 0.730
Adjusted Score: 0.647
Cluster sizes: {0: 294, 1: 1000, 2: 547, 3: 299, 4: 84, 5: 101, 6: 364, 7: 290, 8: 136, 9: 220, 10: 117, 11: 218, 12: 340, 13: 231, 14: 48}
Cluster plot saved to: C:\Users\PC\users_final_demo99999\cluster_plot_n15.png
Cluster profiles saved to: C:\Users\PC\users_final_demo99999/cluster_profiles.csv
